# 2.5D unstructured LES solver on an A100: channel Re_τ 395

The unstructured collocated finite-volume solver in the x-y plane, Fourier in the periodic span, RK3 with a projection per stage, WALE, the whole step on CuPy (`src/upiso25.py`, `device="gpu"`). This notebook: (1) checks the environment, (2) profiles one step on this GPU and prints the run-time table, (3) runs the correctness check, (4) launches the channel Re_τ 395 case, (5) compares it with the Moser–Kim–Mansour 1999 DNS.

The first cell finds the repository root above the notebook's folder (or clones the `unstructured` branch if the notebook was downloaded alone) and makes it the working directory; every path below is relative to it. The first code cell installs what is missing into this kernel (pyamg, cupy-cuda12x, ...); `tools/a100/requirements.txt` and `tools/a100/Dockerfile` do the same outside a notebook. The shell cells (`!python ...`) use the same interpreter as the kernel, so installs made here are visible to them.

In [ ]:
# 0a. Find the repository root (the notebook lives in tools/a100/, and Jupyter's working directory is
#     usually the notebook's own folder), or clone the repo if the notebook was downloaded on its own.
import os, subprocess
def find_root():
    d = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(d, "src", "upiso25.py")): return d
        d = os.path.dirname(d)
    return None
root = find_root()
if root is None:
    print("repository not found above this directory; cloning the unstructured branch ...")
    subprocess.check_call(["git", "clone", "-q", "-b", "unstructured", "--depth", "1", "https://github.com/chandc/PICT-Python.git"]); root = os.path.abspath("PICT-Python")
os.chdir(root); print("working directory:", root)
# which version of the code this is: commit, branch, date, and whether the tree is clean. Compare with
# `git log -1 origin/unstructured` on GitHub; if behind, run  git pull  (or re-clone) and restart the kernel.
def git(*args):
    try: return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL).decode().strip()
    except Exception as e: return f"(git unavailable: {e})"
print("commit  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("branch  :", git("rev-parse", "--abbrev-ref", "HEAD"), "  dirty files:", len(git("status", "--porcelain").splitlines()))
remote = git("ls-remote", "https://github.com/chandc/PICT-Python.git", "refs/heads/unstructured").split()[:1]
head = git("rev-parse", "HEAD")
if remote and not head.startswith(remote[0]):
    # behind (or on another branch): bring this checkout to the tip of origin/unstructured
    print("checkout", head[:7], "is not origin/unstructured", remote[0][:7], "-> updating ...")
    print(git("fetch", "-q", "origin", "unstructured")); print(git("checkout", "-q", "-B", "unstructured", "FETCH_HEAD"))
    head = git("rev-parse", "HEAD"); print("now at  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("GitHub  :", remote[0][:7] if remote else "(offline)", "on origin/unstructured", "" if (remote and head.startswith(remote[0])) else "  <-- STILL different: delete the directory and re-run this cell to clone afresh")
assert any(l.startswith('ap.add_argument("--cfl-max"') for l in open("run_uchannel25.py")), "run_uchannel25.py has no --cfl-max flag: this is an old tree; delete it and re-run this cell"
print("driver  : run_uchannel25.py has --cfl-max  (ok)")
import importlib, sys as _s
for mod in [m for m in list(_s.modules) if m == "src" or m.startswith("src.")]: del _s.modules[mod]      # drop any solver modules a previous run imported from the old tree


## 0b. Google Drive: keep every output outside the session
Hosted runtimes lose their disk when the session ends. This mounts Drive (Colab) and starts a background copier that mirrors this run's outputs (`results/<tag>*`, `results/logs/<tag>.log`, `figures/*re395*`, checkpoints) to `MyDrive/PICT-Python_runs/<tag>/` every 10 minutes and whenever `sync_to_drive()` is called. On a new session the resume cell finds the checkpoint on Drive if the local one is gone.

In [ ]:
import os, shutil, glob, threading, time
tag = "uchan395_96x160x128_wale_cpg"
DRIVE_ROOT = None
try:
    from google.colab import drive                       # Colab
    drive.mount("/content/drive", force_remount=False); DRIVE_ROOT = "/content/drive/MyDrive/PICT-Python_runs"
except Exception as e:
    for cand in (os.path.expanduser("~/Google Drive/My Drive"), "/content/drive/MyDrive"):
        if os.path.isdir(cand): DRIVE_ROOT = os.path.join(cand, "PICT-Python_runs"); break
    if DRIVE_ROOT is None: print("no Google Drive here (not Colab, no local Drive folder):", type(e).__name__, "- outputs stay local")
if DRIVE_ROOT:
    DRIVE_OUT = os.path.join(DRIVE_ROOT, tag); os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(os.path.join(DRIVE_OUT, "logs"), exist_ok=True); print("Drive folder:", DRIVE_OUT)

def sync_to_drive(verbose=True):
    """copy this run's outputs to Drive if newer than the copy there; safe to call any time"""
    if not DRIVE_ROOT: return
    n = 0
    for pat, sub in ((f"results/{tag}*.npz", ""), (f"results/logs/{tag}*.log", "logs"), ("figures/*re395*.png", ""), ("figures/*395*.png", "")):
        for f in glob.glob(pat):
            dst = os.path.join(DRIVE_OUT, sub, os.path.basename(f))
            if not os.path.exists(dst) or os.path.getmtime(f) > os.path.getmtime(dst) + 1:
                shutil.copy2(f, dst); n += 1
    if verbose: print(time.strftime("%H:%M:%S"), f"synced {n} file(s) to Drive")

def _sync_loop(period=600):
    while True:
        time.sleep(period)
        try: sync_to_drive(verbose=False)
        except Exception as e: print("drive sync failed:", e)
if DRIVE_ROOT and not any(t.name == "drive-sync" for t in threading.enumerate()):
    threading.Thread(target=_sync_loop, name="drive-sync", daemon=True).start(); print("background sync every 10 min started")

def restore_from_drive():
    """bring a checkpoint/log back from Drive into results/ (new session, or after the local disk was lost)"""
    if not DRIVE_ROOT: return None
    os.makedirs("results/logs", exist_ok=True); got = []
    for f in glob.glob(os.path.join(DRIVE_OUT, f"{tag}*.npz")) + glob.glob(os.path.join(DRIVE_OUT, "logs", f"{tag}*.log")):
        dst = os.path.join("results", "logs" if f.endswith(".log") else "", os.path.basename(f))
        if not os.path.exists(dst): shutil.copy2(f, dst); got.append(dst)
    print("restored from Drive:", got or "nothing new"); return got
restore_from_drive()

In [ ]:
# 0. Dependencies into THIS kernel (pip of the running interpreter, not the shell's). pyamg builds the
#    multigrid hierarchy on the host; cupy-cuda12x is the GPU array library (binary wheels on x86-64).
import sys, subprocess, importlib
def ensure(mod, pkg):
    try: importlib.import_module(mod); print(f"ok  {mod}")
    except ImportError:
        print(f"installing {pkg} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg]); importlib.invalidate_caches(); importlib.import_module(mod); print(f"ok  {mod} (installed)")
ensure("numpy", "numpy>=1.26"); ensure("scipy", "scipy>=1.11"); ensure("pyamg", "pyamg>=5.0"); ensure("matplotlib", "matplotlib>=3.7")
# CuPy must match the DRIVER's CUDA version, not just be importable: a wheel built for CUDA 13 on a machine
# whose driver supports 12.x imports fine and then fails on the first array with
# "cudaErrorInsufficientDriver". Test a real device operation and reinstall the right wheel if needed.
import re
def cupy_works():
    try:
        import cupy; cupy.zeros(1).sum(); print("ok  cupy", cupy.__version__, "on", cupy.cuda.runtime.getDeviceProperties(0)["name"]); return True
    except Exception as e:
        print("cupy not usable:", type(e).__name__, str(e)[:160]); return False
if not cupy_works():
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout; m_ = re.search(r"CUDA Version: (\d+)\.(\d+)", smi)
    major = int(m_.group(1)) if m_ else 12; pkg = "cupy-cuda12x" if major < 13 else "cupy-cuda13x"
    print(f"driver supports CUDA {m_.group(0) if m_ else '?'} -> installing {pkg} (removing any other cupy build)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "cupy", "cupy-cuda12x", "cupy-cuda13x", "cupy-cuda11x"], capture_output=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    raise SystemExit(f"{pkg} installed. RESTART THE KERNEL/RUNTIME (Runtime > Restart session), then run the cells again from the top.")


In [ ]:
import os, sys, subprocess, time; sys.path.insert(0, os.getcwd())      # cell 0a set the working directory to the repo root
import numpy as np, cupy as cp
p = cp.cuda.runtime.getDeviceProperties(0); print(p["name"], f"{p['totalGlobalMem']/2**30:.0f} GB, {p['multiProcessorCount']} SMs, cupy {cp.__version__}")
import pyamg, scipy; print("pyamg", pyamg.__version__, "scipy", scipy.__version__, "numpy", np.__version__)   # if this fails, rerun cell 0

## 1. Profile one step on this GPU
The size ladder on a periodic box and the run-time table for the target cases at this device's measured rate. The step is memory-bandwidth bound past a ~50 ms launch floor (about 700 kernel launches per step from Python); on the GB10 the box rate was 77–112 ms per 10⁵ cell-modes and a real stretched mesh 256.

In [ ]:
!{sys.executable} tools/a100/profile_step.py --sizes 64x32,128x64,256x64,384x64

## 2. Correctness on this device (2 min)
The 3D Taylor–Green energy balance: −dE/dt must equal the scheme's own discrete dissipation to 0.1–0.2% at every sample (record section 51), E₀ = 31.006277.

In [ ]:
!TGV_DEVICE=gpu TGV_SOLVER=amg TGV_RE=100 TGV_T=2 TGV_N=32 TGV_NZ=32 TGV_DT=0.02 {sys.executable} test_utgv3d.py A

## 3. The channel Re_τ 395

Setup: δ = u_τ = 1, ν = 1/395, constant pressure gradient f_x = 1 (so u_τ = 1 by construction and the measured wall stress is a check), minimal box L_x = π, L_z = 0.34π (L_x⁺ = 1241, L_z⁺ = 422), 96 × 160 wall-clustered quads (Δx⁺ 12.9, Δy⁺ 1.0 at the wall) × 128 Fourier planes (Δz⁺ 3.3), dt 0.001, WALE. Initial condition: the Re_τ 180 DNS field (the same one the 180 case starts from) with its plane mean replaced by the MKM Re_τ 395 mean, so the flow starts turbulent at the right bulk velocity (U_b/u_τ = 17.54) and re-equilibrates over the first ~10 time units. Statistics from t = 10 to 30.

**Time step.** `--cfl-max 0.8` limits the face-flux Courant number (the sum over faces of the outgoing flux over the cell volume, plus |w|/Δz, times dt) by halving dt when needed and restoring it one level per step; RK3 with explicit central convection diverged at a fixed dt 0.001 after ten time units at a Courant number of 1.2 in this measure (2026-09-25 run). Expect dt 0.0005 most of the time, i.e. 60,000 steps.

Cost: 30,000 steps at dt 0.001, 60,000 at 0.0005. Measured on the GB10 at exactly this size: 2.1–2.8 s/step (18–23 h; 1.0M cell-modes at the real-mesh rate of ~230 ms per 10⁵, CFL 0.66). On an A100-80GB expect 0.35–0.75 s/step (3–6 h) from the bandwidth ratio; the profile cell above gives this device's own number. The run is launched in the background and logs to `results/logs/`; the cell below tails the log. Checkpoints every 5000 steps; the cell after the launch resumes from the last checkpoint (`--restart`), which is also how to continue after the diverged run of 2026-09-25 (its checkpoint at t = 10 is valid: the blow-up happened at t = 10.18).

Reference: MKM 1999 at Re_τ = 392.24 (full box 2π × π). A minimal box reproduces the near-wall statistics; the outer-region profiles (y⁺ > ~120) are box-dependent, so judge the log region and the peaks, not the centreline.

In [ ]:
assert any(l.startswith('ap.add_argument("--cfl-max"') for l in open("run_uchannel25.py")), "old driver: re-run cell 0a"
os.makedirs("results/logs", exist_ok=True)
cmd = f"nohup {sys.executable} -u run_uchannel25.py --device gpu --re-tau 395 --nx 96 --ny 160 --nz 128 --dt 0.001 --cfl-max 0.8 --T 30 --t-stats 10 --report 1000 --checkpoint 5000 --tag {tag} > results/logs/{tag}.log 2>&1 &"
print(cmd); subprocess.Popen(cmd, shell=True); time.sleep(60); print(open(f"results/logs/{tag}.log").read()[-1500:])

In [ ]:
# progress: rerun this cell; each report line is 1 time unit
print(open(f"results/logs/{tag}.log").read()[-2500:])
sync_to_drive()

In [ ]:
# resume from the last checkpoint (after a divergence, a restart of the machine, or to extend T): same tag, same log
ck = f"results/{tag}_ckpt.npz"
if not os.path.exists(ck): restore_from_drive()
print("checkpoint:", ck, os.path.exists(ck))
cmd = f"nohup {sys.executable} -u run_uchannel25.py --device gpu --re-tau 395 --nx 96 --ny 160 --nz 128 --dt 0.001 --cfl-max 0.8 --T 30 --t-stats 10 --report 1000 --checkpoint 5000 --tag {tag} --restart {ck} >> results/logs/{tag}.log 2>&1 &"
print(cmd); subprocess.Popen(cmd, shell=True); time.sleep(60); print(open(f"results/logs/{tag}.log").read()[-1200:])

## 4. Compare with MKM 1999
When the log shows `RESULT ...`. Mean profile, rms, Reynolds shear stress in wall units against the DNS; the criteria of the plan (V2): mean within 3% of u_τ in the log region, u_rms peak within 5%, Re_τ within 2%, pressure two-colour mode < 1%.

In [ ]:
import matplotlib.pyplot as plt
d = np.load(f"results/{tag}_stats.npz"); A = np.loadtxt("reference/mkm_chan395/chan395.means"); B = np.loadtxt("reference/mkm_chan395/chan395.reystress")
yd, Ud = A[:, 0], A[:, 2]; ypd = yd * 392.24; ud, vd, wd, uvd = np.sqrt(B[:, 2]), np.sqrt(B[:, 3]), np.sqrt(B[:, 4]), -B[:, 5]
yp, ut = d["yp"], float(d["ut"]); print(f"u_tau {ut:.4f}  Re_tau {float(d['re_tau']):.1f}  samples {int(d['nsamp'])}")
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
ax[0].semilogx(ypd[1:], Ud[1:], "k-", lw=2, label="MKM 1999 Re_tau 392"); ax[0].semilogx(yp, d["U"] / ut, "o-", ms=3, lw=1, label="2.5D LES"); yl = np.logspace(1, 2.5, 50); ax[0].semilogx(yl, np.log(yl) / 0.41 + 5.2, "0.5", ls="--", lw=0.8); ax[0].set(xlabel="y+", ylabel="U+"); ax[0].legend()
for arr, ls, nm in ((ud, "-", "u'"), (vd, "--", "v'"), (wd, ":", "w'")): ax[1].plot(ypd, arr, "k", ls=ls, lw=2, label=f"DNS {nm}")
for k, ls in (("urms", "-"), ("vrms", "--"), ("wrms", ":")): ax[1].plot(yp, d[k] / ut, "C0", ls=ls, lw=1.2)
ax[1].set(xlabel="y+", ylabel="rms / u_tau", xlim=(0, 395)); ax[1].legend()
ax[2].plot(ypd, uvd, "k-", lw=2, label="DNS"); ax[2].plot(yp, -d["uv"] / ut**2, "C0o-", ms=3, lw=1, label="2.5D LES"); ax[2].set(xlabel="y+", ylabel="-<u'v'>/u_tau^2", xlim=(0, 395)); ax[2].legend()
plt.tight_layout(); plt.savefig("figures/uchannel_re395_profiles.png", dpi=130); plt.show()
sel = (yp > 30) & (yp < 120); dU = d["U"][sel] / ut - np.interp(yp[sel], ypd, Ud)
print(f"log region 30<y+<120: U+ - DNS mean {dU.mean():+.3f} (max {np.abs(dU).max():.3f}) u_tau units;  u_rms+ peak {(d['urms']/ut).max():.3f} vs DNS {ud.max():.3f} ({((d['urms']/ut).max()/ud.max()-1)*100:+.1f}%);  -<uv>+ max {(-d['uv']/ut**2).max():.3f} vs {uvd.max():.3f}")
sync_to_drive()

## 5. Notes
* `--forcing mf --Ub 17.54` runs at constant mass flow instead (Re_τ becomes an outcome).
* `--Lx 6.2832 --Lz 3.1416 --nx 192 --nz 256` is the MKM full box (4× the cost).
* Triangle meshes are not for this case: every triangle configuration failed the channel criteria (record section 55).
* The pressure two-colour indicator printed each report line is the mode that broke the structured code's channel; it must stay < 1% of p_rms.